# 05 — Relation extraction

**What you will learn**

- What `POST /extract_relations` returns and how to read `[head, tail]` pairs.
- **Why a boundary checkpoint is what makes relations possible at all** — the
  central idea of this notebook.
- How to get relations in the same forward pass as entities and classification,
  via `schema_config.relations`.
- What `include_confidence` does to the relation response shape, and the
  non-obvious thing the two scores mean.
- Why relation output must be type-validated downstream, with a live example of
  the model getting an argument type wrong.

**What it assumes you already did**

Notebooks [01](01-getting-started.ipynb) through
[04](04-tuning-and-response-shapes.ipynb). In particular you need notebook 01's
`architecture` check and notebook 03's `schema_config` nesting.

**Roughly how long**

About 20 minutes.

## Setup

In [1]:
import json
import os
import time

import requests

# Every notebook in this path reads the same environment variable, so you can
# point the whole series at a different deployment with one export:
#     export GLINER_BASE_URL=http://localhost:8013
BASE_URL = os.environ.get("GLINER_BASE_URL", "http://192.168.1.177:8013")

# The server bounds its own inference at REQUEST_TIMEOUT_SECONDS (default 120)
# and returns 504 when it blows through that. A client timeout slightly above
# the server's means the server always gets to explain itself with a status
# code instead of the client giving up first and leaving you guessing.
TIMEOUT = 130

session = requests.Session()
session.headers.update({"Content-Type": "application/json"})


def get(path):
    """GET a path and return parsed JSON. Raises on non-2xx."""
    r = session.get(f"{BASE_URL}{path}", timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


def post(path, payload):
    """POST JSON and return parsed JSON. Raises on non-2xx."""
    r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


def post_raw(path, payload):
    """POST JSON and return (status_code, parsed_body_or_text). Never raises.

    Used whenever the interesting part of the answer IS the status code.
    """
    r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text


def show(obj):
    """Pretty-print a JSON-serializable object."""
    print(json.dumps(obj, indent=2, ensure_ascii=False))


print("BASE_URL =", BASE_URL)

BASE_URL = http://192.168.1.177:8013


## Why a boundary checkpoint enables relations

Notebook 01 introduced the span/boundary distinction. This is where it earns its
keep, so it is worth going one level deeper than "boundary models can do
relations".

A **span** model works by scoring candidate text spans against your labels
independently. Ask it for `person` and it hands back the fragments that scored
above threshold for "person". Each of those is a scored piece of text, and
crucially, they are scored *in isolation* — there is no step in which the model
decides "these are the entities in this document, and there are exactly four of
them, occupying these positions".

A relation is fundamentally different in kind. `works_for(Satya Nadella,
Microsoft)` is not a property of a span; it is an **edge between two
arguments**. To predict an edge, the model needs two things a span model cannot
give it:

1. **Identified, delimited arguments to connect.** You cannot draw an edge
   between "a fragment that scored 0.9 for person" and "a fragment that scored
   0.8 for company" without first committing to those being *the* entities. A
   bag of independently scored, potentially overlapping fragments has no
   well-defined node set.
2. **A representation of each argument as a unit**, so a head and a tail can be
   scored *against each other*. Span scoring produces label-relevance, not
   pairwise compatibility.

A **boundary** model — the GLiNER2.5 family — predicts entity boundaries as a
first-class step: it decides where mentions start and end, producing a set of
delimited, addressable mentions. That set is exactly the node set a relation
head needs. Once you have nodes, edges become a scoring problem over pairs.

That is the whole reason for the architecture split, and it is why the failure
mode is a clean `501 Not Implemented` rather than a degraded result. There is no
sensible fallback: a span model is not doing relations badly, it lacks the
intermediate representation the task is defined over.

The same structural commitment is what makes efficient batching possible on the
four `*_batch` routes in notebook 06 — hence the same five routes needing the
same checkpoint.

In [2]:
health = get("/health")
ARCH = health["architecture"]
IS_BOUNDARY = ARCH == "boundary"
print("architecture:", ARCH, "| boundary-only routes available:", IS_BOUNDARY)

architecture: boundary | boundary-only routes available: True


If that printed `False`, everything below returns `501`. That is the correct
behavior, not a bug — set `MODEL_ID` to a GLiNER2.5 checkpoint
(`fastino/gliner2.5-base-v1` is the default) and restart the service. The cells
below are written to run either way so you can see the 501 for yourself.

## `POST /extract_relations`

The request is minimal: a `text` and a list of `relations` — the relation type
names you want. Same zero-shot idea as everywhere else in this API; the relation
vocabulary travels with the request.

The response nests under `relation_extraction`, keyed by your relation names,
with each value a list of `[head, tail]` pairs.

In [3]:
RELATION_TEXT = "Satya Nadella, CEO of Microsoft, met Sam Altman of OpenAI in Seattle."
RELATIONS = ["works_for", "met_with", "located_in"]

status, body = post_raw("/extract_relations", {
    "text": RELATION_TEXT,
    "relations": RELATIONS,
})
print("HTTP", status)
show(body)

if status == 501:
    print("\nExpected on a span checkpoint. Set MODEL_ID to a GLiNER2.5 model.")

HTTP 200
{
  "relation_extraction": {
    "works_for": [
      [
        "Satya Nadella",
        "Microsoft"
      ]
    ],
    "met_with": [
      [
        "Satya Nadella",
        "Sam Altman"
      ]
    ],
    "located_in": [
      [
        "Sam Altman",
        "Seattle"
      ]
    ]
  }
}


### Read that output critically

`works_for` and `met_with` look right. Now look at `located_in`.

The model returned a **person** as the head of `located_in` — pairing a human
with a city, rather than the organization you probably had in mind. Is that
wrong? Arguably not: Sam Altman was, in the sentence, in Seattle. It is a
defensible reading of an English sentence.

But it is almost certainly not what your schema meant. If `located_in` is a
column in an `organizations` table, you just got a person in it.

**This is the defining characteristic of relation extraction on this model: it
applies no type constraint you did not give it.** The relation name is a string.
There is no signature, no domain, no range. Nothing anywhere in the request says
"the head of `located_in` must be an organization", so nothing checks it.

You have two levers, and you should use both:

1. **Name relations more specifically.** `org_headquartered_in` carries more
   signal in its embedding than `located_in`, and the model is matching on that
   string. A vaguer name buys you a vaguer relation.
2. **Validate downstream, always.** Extract the entities in the same request
   (next section), then check that each relation argument appears in the entity
   set with the type you expect, and drop or flag the ones that do not. This is
   ten lines of code and it is not optional.

## `include_confidence` on relations

The shape change here is larger than anywhere else in the API. A `[head, tail]`
*array* becomes an object with `head` and `tail` keys, each carrying its own
`text` and `confidence`.

In [4]:
if IS_BOUNDARY:
    show(post("/extract_relations", {
        "text": RELATION_TEXT,
        "relations": ["works_for", "met_with"],
        "include_confidence": True,
    }))
else:
    print("skipped: needs a boundary checkpoint")

{
  "relation_extraction": {
    "works_for": [
      {
        "head": {
          "text": "Satya Nadella",
          "confidence": 0.9527838230133057
        },
        "tail": {
          "text": "Microsoft",
          "confidence": 0.9527838230133057
        }
      }
    ],
    "met_with": [
      {
        "head": {
          "text": "Satya Nadella",
          "confidence": 0.93608158826828
        },
        "tail": {
          "text": "Sam Altman",
          "confidence": 0.93608158826828
        }
      },
      {
        "head": {
          "text": "Satya Nadella",
          "confidence": 0.7093586921691895
        },
        "tail": {
          "text": "OpenAI",
          "confidence": 0.7093586921691895
        }
      }
    ]
  }
}


Note the non-obvious detail: **head and tail carry the same score.**

That is not a coincidence or a rounding artifact. It is the confidence of *the
relation*, reported on both arguments, not two independent per-argument scores.
There is one prediction being made — "this edge exists" — and it has one
probability.

So do not average them, do not treat a high head score with a low tail score as
a meaningful signal (it cannot happen), and do not read the head's score as
"how sure the model is that this is a person". Read either one as the edge's
confidence and ignore the duplication.

The shape change is also a bigger deal for parsing than the entity case in
notebook 04. Entity code goes from a string to a dict — annoying but local.
Relation code goes from `head, tail = pair` (tuple unpacking on a two-element
list) to `pair["head"]["text"], pair["tail"]["text"]`. Those are structurally
different accesses and no amount of duck typing bridges them. Normalize at the
boundary, as in notebook 04.

## Relations in a multi-task request

`schema_config.relations` puts relations in the **same forward pass** as
entities, classification and structure. This is the fourth of the four legal
`schema_config` keys from notebook 03.

This is not just a convenience. It is the single most useful thing in this
notebook, for the reason the previous section set up: **you need the entity set
to validate the relation arguments.** Getting both from one request means they
are guaranteed to come from the same forward pass over the same text — there is
no risk of the two calls disagreeing because of a threshold difference, a
retry, or a model reload in between.

Like the other options here, `schema_config.relations` used to be accepted and
silently dropped. It is honoured now.

In [5]:
status, body = post_raw("/extract_multitask", {
    "text": RELATION_TEXT,
    "schema_config": {
        "entities": ["company", "person", "location"],
        "relations": ["works_for", "met_with", "located_in"],
    },
})
print("HTTP", status)
show(body)

HTTP 200
{
  "entities": {
    "company": [
      "Microsoft",
      "OpenAI"
    ],
    "person": [
      "Satya Nadella",
      "Sam Altman"
    ],
    "location": [
      "Seattle"
    ]
  },
  "relation_extraction": {
    "works_for": [
      [
        "Satya Nadella",
        "Microsoft"
      ]
    ],
    "met_with": [
      [
        "Satya Nadella",
        "Sam Altman"
      ]
    ],
    "located_in": [
      [
        "Sam Altman",
        "Seattle"
      ]
    ]
  }
}


Entities land under `entities`, relations under `relation_extraction` — the same
key the dedicated route uses, so a client can share parsing code between the
two.

And remember the strict validation from notebook 03: `schema_config` accepts
exactly `entities`, `classification`, `relations`, `structure`. Anything else is
a `400` listing the four.

## Validating relation arguments

Here is the ten lines of code that the "no type constraint" problem calls for.
Given entities and relations from one multi-task call, check each relation
argument against the extracted entity set and its type.

Notice this makes the `located_in` problem *visible* rather than letting it into
your database.

In [6]:
def validate_relations(result, expected_types):
    """Check relation arguments against the entity types found in the same pass.

    expected_types: {relation_name: (head_type, tail_type)}
    Returns (accepted, rejected) lists of (relation, head, tail, reason).
    """
    # Build span -> set of entity types, from the same forward pass.
    types = {}
    for label, items in result.get("entities", {}).items():
        for item in items:
            text = item if isinstance(item, str) else item["text"]
            types.setdefault(text, set()).add(label)

    accepted, rejected = [], []
    for rel, pairs in result.get("relation_extraction", {}).items():
        want_head, want_tail = expected_types.get(rel, (None, None))
        for pair in pairs:
            head, tail = (pair if isinstance(pair, list)
                          else (pair["head"]["text"], pair["tail"]["text"]))
            problems = []
            if want_head and want_head not in types.get(head, set()):
                problems.append(f"head {head!r} is {sorted(types.get(head, [])) or 'unknown'}, want {want_head}")
            if want_tail and want_tail not in types.get(tail, set()):
                problems.append(f"tail {tail!r} is {sorted(types.get(tail, [])) or 'unknown'}, want {want_tail}")
            (rejected if problems else accepted).append((rel, head, tail, "; ".join(problems)))
    return accepted, rejected


if status == 200:
    EXPECTED = {
        "works_for":  ("person", "company"),
        "met_with":   ("person", "person"),
        "located_in": ("company", "location"),
    }
    ok, bad = validate_relations(body, EXPECTED)
    print("ACCEPTED")
    for rel, h, t, _ in ok:
        print(f"  {rel}({h}, {t})")
    print("\nREJECTED")
    for rel, h, t, why in bad:
        print(f"  {rel}({h}, {t})  <-- {why}")
else:
    print("skipped: needs a boundary checkpoint")

ACCEPTED
  works_for(Satya Nadella, Microsoft)
  met_with(Satya Nadella, Sam Altman)

REJECTED
  located_in(Sam Altman, Seattle)  <-- head 'Sam Altman' is ['person'], want company


That is the pattern. It is not sophisticated and it does not need to be — it
just needs to exist. Everything it rejects is something that would otherwise
have gone into your store as a fact.

One design note: the validator deliberately uses the entity set from the *same
response*, not an external gazetteer. That keeps it honest about what the model
actually believed on this document, and it means the check works on unseen
entities. If you also have a reference table, run both.

## Try this yourself

Take the same sentence and change only the relation *names* — make them more
specific — and see whether the arguments improve.

Try `["employed_by", "met_in_person_with", "organization_headquartered_in"]`
against the original `["works_for", "met_with", "located_in"]`. Predict first:
does a longer, more specific relation name help, hurt, or do nothing?

In [7]:
if IS_BOUNDARY:
    for names in (["works_for", "met_with", "located_in"],
                  ["employed_by", "met_in_person_with", "organization_headquartered_in"]):
        r = post("/extract_relations", {"text": RELATION_TEXT, "relations": names})
        print("relations:", names)
        show(r)
        print()
else:
    print("skipped: needs a boundary checkpoint")

relations: ['works_for', 'met_with', 'located_in']
{
  "relation_extraction": {
    "works_for": [
      [
        "Satya Nadella",
        "Microsoft"
      ]
    ],
    "met_with": [
      [
        "Satya Nadella",
        "Sam Altman"
      ]
    ],
    "located_in": [
      [
        "Sam Altman",
        "Seattle"
      ]
    ]
  }
}



relations: ['employed_by', 'met_in_person_with', 'organization_headquartered_in']
{
  "relation_extraction": {
    "employed_by": [
      [
        "Satya Nadella",
        "Microsoft"
      ]
    ],
    "met_in_person_with": [
      [
        "Satya Nadella",
        "Sam Altman"
      ]
    ],
    "organization_headquartered_in": [
      [
        "Microsoft",
        "Seattle"
      ],
      [
        "Sam Altman",
        "Seattle"
      ],
      [
        "OpenAI",
        "Seattle"
      ]
    ]
  }
}



**Discussion.** Compare the two outputs carefully, because the result is more
interesting than a clean win.

The specific names **did** help where it counts: `organization_headquartered_in`
returned `["Microsoft", "Seattle"]` and `["OpenAI", "Seattle"]` — actual
organizations as the head, which `located_in` never produced. The relation name
is embedded and matched exactly like an entity label, so putting "organization"
in the string is a real hint about the head's type, and the model used it.

And it **also** still returned `["Sam Altman", "Seattle"]`. The person is right
there in the output alongside the two correct pairs. A more specific name raised
recall on the arguments you wanted without suppressing the one you did not.

That asymmetry is the point of the exercise:

- **Steering changes what the model finds.** It is a cheap experiment with a
  plausible mechanism, and worth running on your own corpus.
- **Steering does not constrain what the model returns.** There is still no
  signature on the relation, so nothing prevents a badly-typed pair from
  appearing. Renaming improved the numerator; it did not close the hole.
- **More recall means more to validate.** Three pairs came back where one did
  before. If you had swapped in the better names and dropped the validator
  because the output "looked right", you would now be writing more wrong facts
  to your store, not fewer.

Be careful about the strength of the conclusion, too: this is one sentence, and
notebook 02's warning about four-sentence comparisons applies in full. An
over-specific name can also under-fire and cost you recall on a different
corpus.

**Do the experiment; keep the validator either way.** Steering improves the
odds. Only the validator makes the output safe to store.

## What you learned

- A boundary checkpoint predicts delimited mentions as a first-class step, which
  gives relation extraction the identified node set it needs. Span models lack
  that intermediate representation entirely, which is why the answer is `501`
  rather than a degraded result.
- `/extract_relations` returns `[head, tail]` pairs under `relation_extraction`,
  keyed by your relation names.
- **No type constraint is applied.** The model returned a person as the head of
  `located_in` on the example sentence. Relation names are just strings with no
  signature.
- `include_confidence` turns each pair into `{"head": {...}, "tail": {...}}`,
  and both carry the **same** score — it is the relation's confidence reported
  twice, not two per-argument scores.
- `schema_config.relations` gets relations and entities from one forward pass,
  which is what makes argument validation reliable: both come from the same pass
  over the same text.
- Validate relation arguments against the entity set every time. Specific
  relation names improve the odds; only the validator makes the output safe.

## Next

**[06 — Batching and throughput](06-batching-and-throughput.ipynb)** — the four
batch routes, the two bounds that constrain them, and why batching helps even
though the GPU is already saturated.